# 📈 Notebook 3 — Evaluation & Visualization (Grade Prediction Version)
### Student Performance Prediction — Multi-Class Edition
**Owner:** Person 3

**Upload before running:**
- From Person 1: `X_test_v2.csv`, `y_test_v2.csv`
- From Person 2: `lr_v2.pkl`, `rf_v2.pkl`, `gb_v2.pkl`, `xgb_v2.pkl`, `label_encoder_v2.pkl`, `results_summary_v2.csv`

## Step 1 — Install & Import

In [ ]:
!pip install xgboost shap -q

import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import shap

from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, f1_score, classification_report
)

sns.set_theme(style='whitegrid')
grade_order = ['A', 'B', 'C', 'D', 'E', 'F']
print('Libraries loaded ✅')

## Step 2 — Upload Files

In [ ]:
from google.colab import files
print('Upload all required files from Person 1 and Person 2')
uploaded = files.upload()

## Step 3 — Load Data & Models

In [ ]:
X_test = pd.read_csv('X_test_v2.csv')
y_test = pd.read_csv('y_test_v2.csv').squeeze()

# Load label encoder — needed to decode XGBoost's numeric predictions back to A/B/C/D/E/F
with open('label_encoder_v2.pkl', 'rb') as f:
    le = pickle.load(f)

# Load all 4 models
model_files = {
    'Logistic Regression': 'lr_v2.pkl',
    'Random Forest':       'rf_v2.pkl',
    'Gradient Boosting':   'gb_v2.pkl',
    'XGBoost':             'xgb_v2.pkl'
}
models = {}
for name, fname in model_files.items():
    with open(fname, 'rb') as f:
        models[name] = pickle.load(f)
    print(f'Loaded {name} ✅')

# Helper: get predictions as letter grades for any model
def get_predictions(name, model):
    if name == 'XGBoost':
        # XGBoost predicts numeric → decode back to letter grade
        return le.inverse_transform(model.predict(X_test))
    return model.predict(X_test)

print(f'\nTest set: {X_test.shape[0]} students')

## Step 4 — Confusion Matrices

In [ ]:
# For multi-class, confusion matrices are 6x6 (one row and column per grade)
# The diagonal = correct predictions
# Off-diagonal = which grades got confused with which
# e.g., B predicted as C is a minor error; B predicted as F is a big error

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

for ax, (name, model) in zip(axes, models.items()):
    y_pred = get_predictions(name, model)
    cm = confusion_matrix(y_test, y_pred, labels=grade_order)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=grade_order)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='weighted')
    ax.set_title(f'{name}\nAcc: {acc:.3f} | F1: {f1:.3f}', fontsize=11, fontweight='bold')

plt.suptitle('Confusion Matrices — Grade Prediction (All Models)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('confusion_matrices_v2.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved confusion_matrices_v2.png ✅')

## Step 5 — Model Comparison Bar Chart

In [ ]:
results = pd.read_csv('results_summary_v2.csv')

x = np.arange(len(results))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 6))
bars1 = ax.bar(x - width/2, results['Accuracy'],    width, label='Accuracy',    color='steelblue',  edgecolor='white')
bars2 = ax.bar(x + width/2, results['F1 Weighted'], width, label='F1 Weighted', color='darkorange', edgecolor='white')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Comparison — Grade Prediction (A/B/C/D/E/F)', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(results['Model'], rotation=15, ha='right', fontsize=10)
ax.set_ylim(0, 1.15)
ax.legend(fontsize=11)
ax.axhline(y=0.5, color='grey', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('model_comparison_v2.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved model_comparison_v2.png ✅')

## Step 6 — Per-Class F1 Score Heatmap

In [ ]:
# This heatmap shows F1 score per grade per model
# Darker = higher F1 = model is better at predicting that specific grade
# Very useful for spotting which grades are hardest to predict (usually A and F
# because they have fewer students in the dataset)

from sklearn.metrics import f1_score as f1

model_names_short = ['LR', 'RF', 'GB', 'XGB']
f1_matrix = []

for name, model in models.items():
    y_pred = get_predictions(name, model)
    # f1 per class: returns one score per grade in grade_order
    scores = f1(y_test, y_pred, labels=grade_order, average=None, zero_division=0)
    f1_matrix.append(scores)

f1_df = pd.DataFrame(f1_matrix, columns=grade_order, index=model_names_short)

plt.figure(figsize=(9, 5))
sns.heatmap(f1_df, annot=True, fmt='.2f', cmap='YlGn',
            linewidths=0.5, vmin=0, vmax=1)
plt.title('F1 Score Per Grade Per Model\n(1.0 = perfect, 0.0 = never predicted correctly)',
          fontsize=12)
plt.xlabel('Grade')
plt.ylabel('Model')
plt.tight_layout()
plt.savefig('f1_per_grade_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved f1_per_grade_heatmap.png ✅')

## Step 7 — SHAP Feature Importance (Best Model)

In [ ]:
# SHAP for multi-class returns one set of values per class
# We average across all classes to get overall feature importance
# This shows which features matter most for predicting ANY grade

best_model_name = results.iloc[0]['Model']
print(f'Running SHAP on best model: {best_model_name}')

rf_model = models['Random Forest']   # use RF as it's most interpretable with SHAP

explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test)

# shap_values is a list of arrays, one per class
# We take the mean absolute value across all classes for overall importance
mean_shap = np.mean([np.abs(sv) for sv in shap_values], axis=0)

# Bar chart of top 15 most important features
feature_importance = pd.Series(
    mean_shap.mean(axis=0),
    index=X_test.columns
).sort_values(ascending=False)

plt.figure(figsize=(10, 7))
feature_importance.head(15).plot(kind='barh', color='steelblue', edgecolor='white')
plt.gca().invert_yaxis()
plt.xlabel('Mean |SHAP Value| (average impact across all grades)')
plt.title('SHAP Feature Importance — Random Forest\n(Top 15 features for Grade Prediction)',
          fontsize=12)
plt.tight_layout()
plt.savefig('shap_importance_v2.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved shap_importance_v2.png ✅')

## Step 8 — Final Summary

In [ ]:
results = pd.read_csv('results_summary_v2.csv')
best = results.iloc[0]

print('=' * 55)
print('    STUDENT GRADE PREDICTION — FINAL RESULTS')
print('=' * 55)
print(f'{"Model":<25} {"Accuracy":>10} {"F1 Weighted":>12}')
print('-' * 55)
for _, row in results.iterrows():
    print(f'{row["Model"]:<25} {row["Accuracy"]:>10.4f} {row["F1 Weighted"]:>12.4f}')
print('=' * 55)
print(f'\n🏆 Best Model   : {best["Model"]}')
print(f'   Accuracy     : {best["Accuracy"]:.4f} ({best["Accuracy"]*100:.1f}%)')
print(f'   F1 Weighted  : {best["F1 Weighted"]:.4f}')
print('\n📁 Output figures:')
for fig in ['confusion_matrices_v2.png', 'model_comparison_v2.png',
            'f1_per_grade_heatmap.png', 'shap_importance_v2.png']:
    print(f'  ✅ {fig}')